In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, VBox, HBox, HTML, Output, Layout
from IPython.display import display, clear_output

# ============================================================
# ABSOLUTE AND RELATIVE QUANTIZATION ERROR
# ============================================================

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.42;
    width:550px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#4f5f9b;
    margin-bottom:8px;
">
Absolute and Relative Quantization Error
</div>

<div style="margin-bottom:5px;">
For a uniform fixed-point quantizer, the quantization step Δ is constant and the absolute error remains bounded by approximately Δ/2.
</div>

<div style="margin-bottom:5px;">
For a floating-point quantizer, the spacing between representable numbers grows with |x|, so the absolute error also tends to increase with signal magnitude.
</div>

<div style="margin-bottom:5px;">
However, this increasing step size approximately preserves relative precision, so |ν|/|x| remains of the same order throughout successive exponent ranges.
</div>

<div>
<b>This notebook:</b> compares absolute and relative quantization errors and shows why floating-point arithmetic provides approximately constant relative accuracy over a wide dynamic range.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}

slider_layout = Layout(
    width='125px',
    min_width='125px'
)

delta_slider = FloatSlider(
    min=0.10,
    max=2.00,
    step=0.10,
    value=0.50,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

p_slider = IntSlider(
    min=2,
    max=10,
    step=1,
    value=4,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

test_slider = FloatSlider(
    min=0.25,
    max=16.00,
    step=0.25,
    value=6.00,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# VALUE LABELS
# ============================================================

value_layout = Layout(
    width='55px',
    min_width='55px',
    margin='0px 0px 0px 4px'
)

delta_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.50</div>',
    layout=value_layout
)

p_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">4</div>',
    layout=value_layout
)

test_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">6.00</div>',
    layout=value_layout
)

# ============================================================
# LABELS
# ============================================================

label_layout = Layout(
    width='155px',
    min_width='155px'
)

delta_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Uniform step Δ:</div>',
    layout=label_layout
)

p_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Significand bits p:</div>',
    layout=label_layout
)

test_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Test amplitude:</div>',
    layout=label_layout
)

# ============================================================
# CONTROL ROWS
# ============================================================

row_layout = Layout(
    width='390px',
    min_width='390px',
    height='38px',
    min_height='38px',
    align_items='center',
    overflow='visible'
)

delta_row = HBox(
    [
        delta_label,
        delta_slider,
        delta_value
    ],
    layout=row_layout
)

p_row = HBox(
    [
        p_label,
        p_slider,
        p_value
    ],
    layout=row_layout
)

test_row = HBox(
    [
        test_label,
        test_slider,
        test_value
    ],
    layout=row_layout
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#4f5f9b;
            margin-bottom:8px;
        ">
        Error Parameters
        </div>
        """),

        delta_row,
        p_row,
        test_row
    ],
    layout=Layout(
        width='420px',
        min_width='420px',
        padding='10px 12px 12px 12px',
        border='1px solid #c3cae2',
        overflow='visible'
    )
)

# ============================================================
# TOP TWO-COLUMN LAYOUT
# ============================================================

top_layout = HBox(
    [
        documentation,
        controls_card
    ],
    layout=Layout(
        width='1020px',
        align_items='flex-start',
        justify_content='space-between',
        gap='16px',
        margin='0px 0px 10px 0px',
        overflow='visible'
    )
)

# ============================================================
# OUTPUT AREAS
# ============================================================

graph_output = Output(
    layout=Layout(
        width='1080px',
        overflow='hidden'
    )
)

result_html = HTML()

# ============================================================
# UNIFORM QUANTIZER
# ============================================================

def uniform_quantize(x, delta):

    return delta * np.floor(
        x / delta + 0.5
    )

# ============================================================
# FLOATING-POINT QUANTIZER
#
# For:
#
# 2^k <= |x| < 2^(k+1)
#
# local spacing:
#
# D = 2^(k-p+1)
# ============================================================

def floating_quantize(x, p):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.zeros_like(
        x
    )

    nonzero = (
        x != 0
    )

    magnitude = np.abs(
        x[nonzero]
    )

    exponent = np.floor(
        np.log2(magnitude)
    ).astype(int)

    step = 2.0 ** (
        exponent - p + 1
    )

    y[nonzero] = np.sign(
        x[nonzero]
    ) * np.round(
        magnitude / step
    ) * step

    return y

# ============================================================
# LOCAL FLOATING-POINT STEP
# ============================================================

def floating_step(x, p):

    x = np.asarray(
        x,
        dtype=float
    )

    step = np.zeros_like(
        x
    )

    nonzero = (
        x != 0
    )

    exponent = np.floor(
        np.log2(np.abs(x[nonzero]))
    ).astype(int)

    step[nonzero] = 2.0 ** (
        exponent - p + 1
    )

    return step

# ============================================================
# MAIN PLOT FUNCTION
# ============================================================

def plot_error_comparison(delta, p, test_amplitude):

    # --------------------------------------------------------
    # FIXED DISPLAY RANGE
    # --------------------------------------------------------

    xmax = 16.0

    x = np.linspace(
        -xmax,
        xmax,
        8000
    )

    # Avoid exact zero for relative-error calculation
    x_positive = np.logspace(
        -2,
        np.log10(xmax),
        5000
    )

    # --------------------------------------------------------
    # UNIFORM QUANTIZATION
    # --------------------------------------------------------

    y_uniform = uniform_quantize(
        x,
        delta
    )

    error_uniform = y_uniform - x

    # --------------------------------------------------------
    # FLOATING-POINT QUANTIZATION
    # --------------------------------------------------------

    y_float = floating_quantize(
        x,
        p
    )

    error_float = y_float - x

    # --------------------------------------------------------
    # POSITIVE AXIS FOR RELATIVE ERROR
    # --------------------------------------------------------

    y_uniform_positive = uniform_quantize(
        x_positive,
        delta
    )

    y_float_positive = floating_quantize(
        x_positive,
        p
    )

    error_uniform_positive = (
        y_uniform_positive - x_positive
    )

    error_float_positive = (
        y_float_positive - x_positive
    )

    relative_uniform = np.abs(
        error_uniform_positive
    ) / x_positive

    relative_float = np.abs(
        error_float_positive
    ) / x_positive

    # --------------------------------------------------------
    # THEORETICAL LOCAL ERROR BOUNDS
    # --------------------------------------------------------

    float_step_positive = floating_step(
        x_positive,
        p
    )

    uniform_absolute_bound = (
        delta / 2.0
    )

    float_absolute_bound = (
        float_step_positive / 2.0
    )

    uniform_relative_bound = (
        delta / (2.0 * x_positive)
    )

    float_relative_bound = (
        float_absolute_bound / x_positive
    )

    # --------------------------------------------------------
    # TEST AMPLITUDE
    # --------------------------------------------------------

    x_test = np.array(
        [test_amplitude]
    )

    uniform_test_output = uniform_quantize(
        x_test,
        delta
    )[0]

    float_test_output = floating_quantize(
        x_test,
        p
    )[0]

    uniform_test_error = (
        uniform_test_output - test_amplitude
    )

    float_test_error = (
        float_test_output - test_amplitude
    )

    uniform_test_relative = (
        abs(uniform_test_error) / test_amplitude
    )

    float_test_relative = (
        abs(float_test_error) / test_amplitude
    )

    local_float_step = floating_step(
        x_test,
        p
    )[0]

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(10.6, 7.3)
    )

    gs = fig.add_gridspec(
        2,
        2,
        hspace=0.46,
        wspace=0.30
    )

    ax1 = fig.add_subplot(
        gs[0, 0]
    )

    ax2 = fig.add_subplot(
        gs[0, 1]
    )

    ax3 = fig.add_subplot(
        gs[1, 0]
    )

    ax4 = fig.add_subplot(
        gs[1, 1]
    )

    # ========================================================
    # GRAPH 1:
    # UNIFORM ABSOLUTE ERROR
    # ========================================================

    ax1.plot(
        x,
        error_uniform,
        linewidth=1.2
    )

    ax1.axhline(
        uniform_absolute_bound,
        linestyle='--',
        linewidth=1.0,
        label='+Δ/2'
    )

    ax1.axhline(
        -uniform_absolute_bound,
        linestyle='--',
        linewidth=1.0,
        label='−Δ/2'
    )

    ax1.axhline(
        0,
        linewidth=0.8
    )

    ax1.axvline(
        test_amplitude,
        linestyle=':',
        linewidth=1.0
    )

    ax1.set_xlim(
        -16,
        16
    )

    ax1.set_ylim(
        -4.2,
        4.2
    )

    ax1.set_xlabel(
        'Input amplitude x',
        fontsize=10
    )

    ax1.set_ylabel(
        'Quantization error',
        fontsize=10
    )

    ax1.set_title(
        'Uniform Quantizer: Absolute Error',
        fontsize=12,
        pad=8
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax1.legend(
        loc='upper right',
        fontsize=8
    )

    # ========================================================
    # GRAPH 2:
    # FLOATING-POINT ABSOLUTE ERROR
    # ========================================================

    ax2.plot(
        x,
        error_float,
        linewidth=1.2
    )

    ax2.axhline(
        0,
        linewidth=0.8
    )

    ax2.axvline(
        test_amplitude,
        linestyle=':',
        linewidth=1.0
    )

    ax2.set_xlim(
        -16,
        16
    )

    ax2.set_ylim(
        -4.2,
        4.2
    )

    ax2.set_xlabel(
        'Input amplitude x',
        fontsize=10
    )

    ax2.set_ylabel(
        'Quantization error',
        fontsize=10
    )

    ax2.set_title(
        'Floating-Point Quantizer: Absolute Error',
        fontsize=12,
        pad=8
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    # ========================================================
    # GRAPH 3:
    # RELATIVE ERROR
    # ========================================================

    ax3.loglog(
        x_positive,
        relative_uniform,
        linewidth=1.3,
        label='Uniform quantizer'
    )

    ax3.loglog(
        x_positive,
        relative_float,
        linewidth=1.3,
        label='Floating-point quantizer'
    )

    ax3.axhline(
        2.0 ** (-p),
        linestyle='--',
        linewidth=1.3,
        label='2⁻ᵖ reference'
    )

    ax3.axvline(
        test_amplitude,
        linestyle=':',
        linewidth=1.0
    )

    ax3.set_xlim(
        1e-2,
        16
    )

    ax3.set_ylim(
        1e-5,
        2
    )

    ax3.set_xlabel(
        'Signal magnitude |x|',
        fontsize=10
    )

    ax3.set_ylabel(
        'Relative error |ν| / |x|',
        fontsize=10
    )

    ax3.set_title(
        'Relative Quantization Error',
        fontsize=12,
        pad=8
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        which='major',
        linestyle=':',
        alpha=0.4
    )

    ax3.legend(
        loc='lower left',
        fontsize=8
    )

    # ========================================================
    # GRAPH 4:
    # THEORETICAL RELATIVE ERROR BOUNDS
    # ========================================================

    ax4.loglog(
        x_positive,
        uniform_relative_bound,
        linewidth=1.8,
        label='Uniform bound Δ/(2|x|)'
    )

    ax4.loglog(
        x_positive,
        float_relative_bound,
        linewidth=1.8,
        label='Floating-point local bound'
    )

    ax4.axhline(
        2.0 ** (-p),
        linestyle='--',
        linewidth=1.2,
        label='Maximum ≈ 2⁻ᵖ'
    )

    ax4.axvline(
        test_amplitude,
        linestyle=':',
        linewidth=1.0
    )

    ax4.set_xlim(
        1e-2,
        16
    )

    ax4.set_ylim(
        1e-5,
        30
    )

    ax4.set_xlabel(
        'Signal magnitude |x|',
        fontsize=10
    )

    ax4.set_ylabel(
        'Relative-error bound',
        fontsize=10
    )

    ax4.set_title(
        'Relative-Precision Bounds',
        fontsize=12,
        pad=8
    )

    ax4.tick_params(
        axis='both',
        labelsize=9
    )

    ax4.grid(
        True,
        which='major',
        linestyle=':',
        alpha=0.4
    )

    ax4.legend(
        loc='lower left',
        fontsize=8
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.93,
        bottom=0.09
    )

    plt.show()

    plt.close(fig)

    # ========================================================
    # RESULT BOX
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.55;
        width:980px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Selected input amplitude:</b>
    |x| = {test_amplitude:.2f}

    <br>

    <b>Uniform quantizer:</b>
    Q(x) = {uniform_test_output:.5f}
    &nbsp;&nbsp;&nbsp;
    ν = {uniform_test_error:.5f}
    &nbsp;&nbsp;&nbsp;
    |ν|/|x| = {uniform_test_relative:.5e}

    <br>

    <b>Floating-point quantizer:</b>
    Q(x) = {float_test_output:.5f}
    &nbsp;&nbsp;&nbsp;
    ν = {float_test_error:.5f}
    &nbsp;&nbsp;&nbsp;
    |ν|/|x| = {float_test_relative:.5e}

    <br>

    <b>Local floating-point spacing:</b>
    D = {local_float_step:.5f}

    &nbsp;&nbsp;&nbsp;

    <b>Maximum relative-error scale:</b>
    2<sup>−p</sup> = {2.0**(-p):.5e}

    </div>
    """

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update_notebook(change=None):

    delta_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {delta_slider.value:.2f}
    </div>
    """

    p_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {p_slider.value}
    </div>
    """

    test_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {test_slider.value:.2f}
    </div>
    """

    with graph_output:

        clear_output(
            wait=True
        )

        plot_error_comparison(
            delta_slider.value,
            p_slider.value,
            test_slider.value
        )

# ============================================================
# CONNECT CONTROLS
# ============================================================

delta_slider.observe(
    update_notebook,
    names='value'
)

p_slider.observe(
    update_notebook,
    names='value'
)

test_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.45;
    width:1080px;
    padding:11px 15px;
    border:1px solid #c8cee5;
    background:#f8f9fe;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#4f5f9b;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The uniform quantizer maintains an approximately constant absolute-error bound ±Δ/2, but its relative error becomes increasingly important as the signal magnitude decreases.
</div>

<div style="margin-bottom:4px;">
In floating-point quantization, the absolute-error scale increases whenever |x| enters a higher exponent range because the spacing between representable numbers increases.
</div>

<div>
This increase is approximately proportional to signal magnitude, so the relative error remains bounded at roughly the scale 2<sup>−p</sup>. This approximately constant relative precision is one of the principal advantages of floating-point representation over a wide dynamic range.
</div>

</div>
""")

# ============================================================
# INITIAL DRAW
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        top_layout,
        graph_output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1080px',
        overflow='visible'
    )
)

display(main_layout)